# Dunnhumby seed48 · 적응형 M2 + M3 + M4-C

- 기존 개발 seed48의 M1·M3+M4-C 완료 결과를 재사용합니다.
- 새 완전결합 **1개만** 처음부터 300epoch 공동학습합니다.
- 학습 DAY1~683 / 개발평가 DAY684~690. 마지막 주 test와 holdout은 사용하지 않습니다.
- 기존 적응형 M2, 두 축 중심화 M3(CV=.20), M4-C(lambda=.5)의 구조를 유지합니다.
- 25epoch 간격 개발곡선은 진단용이며 비교·판정은 고정300epoch입니다.
- 같은 런타임에서는 마지막 epoch, 새 런타임에서는 Drive의 마지막25epoch 복구점부터 재개합니다.
- 완료 결과가 있으면 학습·재평가 없이 다시 읽습니다. L4 GPU를 선택하세요.


In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess, sys, json, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임(L4)을 연결하세요.'
SOURCE_COMMIT = '89922a041a3d14e3e7416a778db8e418f5e7efb9'
REPO = Path('/content/clv-m5-adaptive-nv-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','-q','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','-q','origin'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
sys.path.insert(0,str(REPO))
print('GPU:',torch.cuda.get_device_name(0),'| source:',SOURCE_COMMIT)


In [ ]:
import clv_m5_adaptive_nv_m3_m4c_seed48 as screen
cfg, prepared = screen.prepare()
print('재사용 완료 결과:', cfg.reference_json)
print('새 결과 저장:', cfg.out_dir)
print('M2 초기 계수:', prepared['preflight']['dataset_initialization'])
print('M3/M4 동일성 확인 완료. 다음 셀에서 새 결합 1개만 학습합니다.')


In [ ]:
result = screen.run(cfg, prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


In [ ]:
from IPython.display import display
metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10',
           'price_purchase_amount_weighted_hit@20','vndcg@20',
           'price_purchase_amount_weighted_hit@50','vndcg@50']
at = result['absolute'].loc[result['absolute'].epoch.eq(300)]
display(at.set_index('model_id')[metrics].T)
comparison = result['comparison']
primary = comparison[comparison.epoch.eq(300) & comparison.reference.eq(screen.REFERENCE)]
display(primary.set_index('metric').loc[metrics, ['reference_value','candidate_value','relative_change_pct']])
print('전체142지표와 저/중/고CLV·노출·불리한 결과는 CSV/JSON에 저장합니다.')


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
output = Path('/content/clv_m5_adaptive_nv_m3_m4c_seed48_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
files.download(str(output))
